# Proyecto Circuitos Hamiltonianos — Farmacias Simán SPS

Notebook organizado por etapas.
**Estructura:**
1. Configuración y datos
2. Datos de los candidatos
3. Carga y validación de los 5 bloques
4. Construcción del grafo
5. Carga de conexiones vecinas y distancias de los candidatos
6. Verificación de Dirac y Ore
7. Backtracking del circuito Hamiltoniano
8. Resultados de los circuitos Hamiltonianos
9. Comparación de conectividad de los 3 candidatos


## 1. Importaciones y funciones auxiliares

In [1]:
from pathlib import Path
from math import inf
import heapq
import openpyxl

ARCHIVO_EXCEL=Path("Datos_SucursalesSimán.xlsx")

#convierte el valor del excel en float
def numero(valor):
    if valor is None:
        return None
    
    try:
        
        return float(valor)
    
    except(TypeError,ValueError):
        return None
    
    
def formato_km(valor):
    if valor is None or valor==inf:
        return "N/D"
    
    return f"{valor:.2f} km"

## 2. Datos de los candidatos

In [2]:
CANDIDATOS={
    
    "A": {
        "nombre": "Candidato A",
        "zona": "Zona Norte",
        "ubicacion": "Sector La 105 Brigada",
    },
    "B": {
        "nombre": "Candidato B",
        "zona": "Zona Sur",
        "ubicacion": "Zona industrial, Sector el Polvorín",
    },
    "C": {
        "nombre": "Candidato C",
        "zona": "Zona Centro",
        "ubicacion": "Barrio Los Andes, 21102 San Pedro Sula, Cortés, Honduras",
    },
    
}


def nombre_candidato(letra):
    
    return CANDIDATOS[str(letra).upper()]["nombre"]

def datos_candidato(letra):
    return CANDIDATOS[str(letra).upper()]

if not ARCHIVO_EXCEL.exists():
    
    raise FileNotFoundError(f"No se encontro el archivo: {ARCHIVO_EXCEL.resolve()}")

wb=openpyxl.load_workbook(ARCHIVO_EXCEL,data_only=True)

## 3. Carga y validación de los 5 bloques

In [3]:
#leer los 5 bloques, Lee la hoja 'Bloques de Sucursales' dentro del excel
def cargar_bloques(hoja):
    
    bloques={"Bloque 1":[],"Bloque 2":[],"Bloque 3":[],"Bloque 4":[],"Bloque 5":[],}
    
    #bloques 1,2,3
    for fila in range(3,15):
        columnas=[(2,"Bloque 1"),(5,"Bloque 2"),(8,"Bloque 3"),]
        
        for columna_nombre,bloque in columnas:
            nombre=hoja.cell(fila,columna_nombre).value
            if nombre:
                bloques[bloque].append(str(nombre).strip())
                
    #bloque 4,5
    for fila in range(19,31):
        columnas=[(2,"Bloque 4"),(5,"Bloque 5")]
        
        for columna_nombre,bloque in columnas:
                    nombre=hoja.cell(fila,columna_nombre).value
                    if nombre:
                        bloques[bloque].append(str(nombre).strip())
                        
    return bloques
        
        
hoja_bloques=wb["Bloques de Sucursales"]
bloques=cargar_bloques(hoja_bloques)

#verificar q cada bloque debe tener 12 farmacias 
for bloque,farmacias in bloques.items():
    
    if len(farmacias)!=12:
        raise ValueError(f"{bloque} tiene {len(farmacias)} farmacias; se esperan 12")
    

## 4. Construcción del grafo

In [4]:
#Aca se construyen el grafo de las distancias vecinas 
#Se guarda como conexion no dirigida porque el peso de Dijkstra sera el promedio ida/vuelta. Esto permite que todas las farmacias del bloque sean alcanzables.    

grafo={}

def agregar_vertice(nombre):
    if nombre not in grafo:
        grafo[nombre]={}
    
#Agrega una conexion con pesos ida, vuelta y promedio    
def agregar_conexion(a,b,ida,vuelta):
    
    ida=numero(ida)
    vuelta=numero(vuelta)
    
    if ida is None or vuelta is None:
        return
    
    promedio=(ida+vuelta)/2
    agregar_vertice(a)
    agregar_vertice(b)
    
    datos={"ida":ida,"vuelta":vuelta,"peso":promedio}
    
    #si la misma conexion apareciera mas de una vez, conservarla de menor promedio.
    anterior=grafo[a].get(b)
    if anterior is None or promedio<anterior["peso"]:
        
        grafo[a][b]=datos
        #A -> B
        #ida = 2.0
        #vuelta = 3.0

        #B -> A
        #ida = 3.0
        #vuelta = 2.0
        grafo[b][a]={"ida":vuelta,"vuelta":ida,"peso":promedio}
        
        

## 5. Carga de conexiones vecinas y distancias de los candidatos

In [5]:
hojas_vecinos=wb["Distancias de Farmacias Vecinas"]
farmacia_actual=None

for fila in range(2,hojas_vecinos.max_row+1):
    numero_farmacia=hojas_vecinos.cell(fila,1).value
    nombre=hojas_vecinos.cell(fila,2).value
    
    #una fila con numero de farmacia marca el inicio de un grupo
    if numero_farmacia is not None and nombre:
        farmacia_actual=str(nombre).strip()
        agregar_vertice(farmacia_actual)
        continue
    
    #las siguientes filas contienen sus 3 farmacias vecinas
    if farmacia_actual and nombre:
        vecino=str(nombre).strip()
        ida=hojas_vecinos.cell(fila,4).value
        vuelta=hojas_vecinos.cell(fila,5).value
        
        agregar_conexion(farmacia_actual,vecino,ida,vuelta)

#Distancia de cada candidato hacia la sucursal de entrada de cada bloque.
#No se agregan como vertices del grafo: ya no se usan para Dijkstra,
#solo para comparar directamente la conectividad de los 3 candidatos (seccion 9).
Candidato_distancias={"A":{},"B":{},"C":{}}
hoja_candidatos=wb["Distancias de Candidatos"]

candidato_actual=None

for fila in range(2,17):
    letra=hoja_candidatos.cell(fila,1).value
    bloque=hoja_candidatos.cell(fila,2).value
    sucursal=hoja_candidatos.cell(fila,3).value
    ida=hoja_candidatos.cell(fila,4).value
    vuelta=hoja_candidatos.cell(fila,5).value
    
    if letra:
        candidato_actual=str(letra).strip().upper()
        
    if candidato_actual not in Candidato_distancias:
        continue
    
    if bloque is None or sucursal is None:
        continue
    
    ida=numero(ida)
    vuelta=numero(vuelta)
    
    if ida is None or vuelta is None:
        continue
    
    bloque=int(bloque)
    sucursal=str(sucursal).strip()
    Candidato_distancias[candidato_actual][bloque]={"sucursal":sucursal,"ida":ida,"vuelta":vuelta,"promedio":(ida+vuelta)/2}


## 6. Verificación de Dirac y Ore

# Definicion

1. **Dirac:** 
Es una de las condiciones que nos ayuda a determinar si un grafo tiene circuito hamiltoniano. Eso si que un grafo no cumpla Dirac NO significa necesariamente que no sea hamiltoniano.

    El teorema dice: Para un grafo simple con n ≥ 3 vertices, si el grado de cada vertice cumple: deg(v)≥n/2

    En otras palabras: Si cada vertice esta conectado con al menos la mitad de los demas vertices, entonces existe un circuito hamiltoniano.

    Ejem: vertices(n)=6  6/2=3  -> para todos los vertices si tenemos A:4 | B: 3 | C:5 | D:3 | E:4 | F::3
    El grafo es un **circuito hamiltoniano**


2. **Ore:**
Ore utiliza una condicion diferente.

    Para un grafo simple con n≥3, si para cada par de vértices no adyacentes (u) y (v): deg(u)+deg(v) ≥ n
    Entonces el grafo tiene **circuito hamiltoniano**

    No adyacentes: Significa que no existe una arista entre ellos.  

    Ejem: n=6. Para aplicar Ore necesitamos revisar los pares de vertices que no estan conectados.
    Suponiendo tenemos deg(A)=4 y deg(D)=3 y A y D  no estan conectados.
    Entonces  deg(A)+deg(D)=4+3=7 -> como 7≥6, se cumple la condicion la condicion para ese par.
    Hay que hacer esto con todos los pares de vértices no adyacentes.
    deg(u)+deg(v)≥n

    entonces:

    Existe un circuito hamiltoniano
	​
    Si todos cumplen:
    deg(u)+deg(v)≥n
    Entonces **Existe un circuito hamiltoniano**

                                                                     





In [6]:
#VERIFICACION de condiciciones hamiltonianas


#Obtiene unicamente las farmacias pertenecientes al bloque.  El candidato NO forma parte del circuito Hamiltoniano.

def obtener_subgrafo_bloque(bloque):
    nodos=set(bloques[bloque])
    subgrafo={nodo:{}for nodo in nodos}
    for u in nodos:
        for v,datos in grafo.get(u,{}).items():
            if v in nodos:
                subgrafo[u][v]=datos
                
    return subgrafo

#Calcula el grado de cada vertice del subgrafo.
def calcular_grados(subgrafo):
    grados={}
    for vertice in subgrafo:
        grados[vertice]=len(subgrafo[vertice])
        
    return grados

#Verifica la condicion suficiente de Dirac. Un grafo simple con n >= 3 vertices cumple Dirac si:grado minimo >= n / 2
#si se cumple, el grafo es Hamiltoniano.
def verificar_dirac(subgrafo):
    n=len(subgrafo)
    grados=calcular_grados(subgrafo)
    
    if n<3:
        return{"cumple":False,"n":n,"grado_minimo":min(grados.values())if grados else 0}
    
    grado_minimo=min(grados.values())
    cumple=grado_minimo>=n/2
    
    return{"cumple":cumple,"n":n,"grado_minimo":grado_minimo}

#Verifica la condicion suficiente de Ore.  Para todo par de vertices NO adyacentes:  grado(u) + grado(v) >= n
def verificar_ore(subgrafo):
    
    n=len(subgrafo)
    grados=calcular_grados(subgrafo)
    pares_evaluados=[]
    cumple=True
    vertices=list(subgrafo.keys())
    for i in range(len(vertices)):
        for j in range(i+1,len(vertices)):
            u=vertices[i]
            v=vertices[j]
            
            #solo interesa revisar pares no conectados
            if v not in subgrafo[u]:
                suma_grados=grados[u]+grados[v]
                
                pares_evaluados.append((u,v,grados[u],grados[v],suma_grados))
                
                if suma_grados<n:
                    cumple=False

    return{"cumple":cumple,"n":n,"pares":pares_evaluados}


#10 mostrar resultados
def imprimir_linea():# XD
    print("="*90)
    
def imprimir_verificacion_hamiltoniana(numero_bloque):
    
    bloque=f"Bloque {numero_bloque}"
    subgrafo=obtener_subgrafo_bloque(bloque)
    
    grados=calcular_grados(subgrafo)
    resultado_dirac=verificar_dirac(subgrafo)
    resultado_ore=verificar_ore(subgrafo)
    
    print("\n"+"="*90)
    print(f"VERIFICACIÓN HAMILTONIANA - {bloque.upper()}")
    print("=" * 90)
    
    print("\nFarmacias del bloque:")
    
    for vertice in sorted(grados):
        print(f"  {vertice}: grado = {grados[vertice]}")
    
    print("\n" + "-" * 90)
    print("TEOREMA DE DIRAC")
    print("-" * 90)
    
    print(f"Numero de vertices: {resultado_dirac['n']}")
    print(f"Grado minimo:       {resultado_dirac['grado_minimo']}")
    print(f"n / 2:              {resultado_dirac['n'] / 2:.1f}")
    
    if resultado_dirac["cumple"]:
        print("Resultado: CUMPLE LA CONDICION DE DIRAC")
        print("Conclusion: el bloque tiene garantizada la existencia de")
        print("un circuito Hamiltoniano mediante el Teorema de Dirac.")
    else:
        print("Resultado: NO CUMPLE LA CONDICION DE DIRAC")
        print("Esto NO significa que no exista un circuito Hamiltoniano.")
    
    print("\n" + "-" * 90)
    print("TEOREMA DE ORE")
    print("-" * 90)
    
    if resultado_ore["cumple"]:
        print("Resultado: CUMPLE LA CONDICION DE ORE")
        print("Conclusión: el bloque tiene garantizada la existencia de")
        print("un circuito Hamiltoniano mediante el Teorema de Ore.")
    else:
        print("Resultado: NO CUMPLE LA CONDICION DE ORE")
        print("Esto NO significa que no exista un circuito Hamiltoniano.")
        
        

## 7. Backtracking para el circuito Hamiltoniano

In [7]:
# 11 busqueda 
# Busca un circuito Hamiltoniano utilizando backtracking.

#    Un circuito Hamiltoniano debe:
#    1. Visitar todos los vértices exactamente una vez.
#    2. No repetir vertices durante el recorrido.
#    3. Regresar al vértice inicial.
    
def buscar_circuito_hamiltoniano(subgrafo):
    vertices=list(subgrafo.keys())
    if len(vertices)<3:
        return[]
    
    #se elige una farmacia como punto inicial
    inicio=vertices[0]
    ruta=[inicio]
    visitados={inicio}
    
    
    def backtracking(actual):
        
        # Si ya visitamos todos los vértices,comprobamos si podemos regresar al inicio.
        if len(ruta)==len(vertices):
            if inicio in subgrafo[actual]:
                return True
            
            return False
        
        #se revisa los vecinos del vertice actual
        for vecino in subgrafo[actual]:
            
            #si la farmacia todavia no ha sido visitada
            if vecino not in visitados:
                
                #agregar a la ruta
                ruta.append(vecino)
                visitados.add(vecino)
                
                #continuar buscando
                if backtracking(vecino):
                    return True
                
                #si esta opcion no funciona, se deshace la decision
                ruta.pop()
                visitados.remove(vecino)
            
        return False
    
    #iniciar busqueda
    encontrado=backtracking(inicio)
    
    if encontrado:
        #se cierra el circuito regresando al inicio
        ruta.append(inicio)
        return ruta
    
    return[]


## 8. Verificación y distancia del circuito Hamiltoniano

In [8]:
#Verifica que un circuito encontrado sea Hamiltoniano.
def verificar_circuito_hamiltoniano(circuito,subgrafo):
    
    if not circuito:
        return False
    
    #El primer y ultimo vértice deben ser iguales
    if circuito[0]!=circuito[-1]:
        return False
    
    #Quito el ultimo elemento para contar solamente los vertices visitados.
    vertices_visitados=circuito[:-1]
    
    #Todos los vertices deben ser diferentes
    if len(vertices_visitados) != len(set(vertices_visitados)):
        return False
    
    #se debe visitar todos los vertices del bloque
    if set(vertices_visitados)!=set(subgrafo.keys()):
        return False
    
    #se verifica que cada conexion exista
    for actual,siguiente in zip(circuito,circuito[1:]):
        if siguiente not in subgrafo[actual]:
            return False
        
    return True


#  Calcula la distancia total de ida, vuelta y promedio del circuito Hamiltoniano.
def calcular_distancia_circuito(circuito):
    
    total_ida=0.0
    total_vuelta=0.0
    
    for actual,siguiente in zip(circuito,circuito[1:]):
        datos=grafo[actual][siguiente]
        total_ida+=datos["ida"]
        total_vuelta+=datos["vuelta"]
        
    promedio=(total_ida+total_vuelta)/2
    
    return total_ida,total_vuelta,promedio


def analizar_circuitos_hamiltonianos():

    resultados={}

    print("\n")
    imprimir_linea()
    print("CIRCUITOS HAMILTONIANOS DE LOS 5 BLOQUES")
    imprimir_linea()

    for numero_bloque in range(1, 6):

        bloque=f"Bloque {numero_bloque}"
        subgrafo=obtener_subgrafo_bloque(bloque)
        circuito=buscar_circuito_hamiltoniano(subgrafo)

        print("\n" + "=" * 90)
        print(f"{bloque.upper()}")
        print("=" * 90)

        if not circuito:
            print("\nNo se encontro un circuito Hamiltoniano.")
            resultados[numero_bloque]={"circuito": [],"valido": False}
            continue

        valido=verificar_circuito_hamiltoniano(circuito,subgrafo)

        print("\nCircuito encontrado:")

        print(" -> ".join(circuito))

        print("\nCantidad de vértices visitados:")

        print(len(circuito)-1)

        print("\n¿Es un circuito Hamiltoniano?")

        if valido:
            print("SÍ")
        else:
            print("NO")

        if valido:

            ida,vuelta,promedio=calcular_distancia_circuito(circuito)

            print("\nDistancia del circuito:")
            print(f"  Ida:      {ida:.2f} km")
            print(f"  Vuelta:   {vuelta:.2f} km")
            print(f"  Promedio: {promedio:.2f} km")

            resultados[numero_bloque]={"circuito": circuito,"valido": True,"ida": ida,"vuelta": vuelta,"promedio": promedio}

    return resultados

    

## 9. Ejecutar verificación y búsqueda Hamiltoniana

Primero verificamos Dirac y Ore para los cinco bloques y después buscamos un circuito Hamiltoniano mediante backtracking.

> Si un bloque no cumple Dirac u Ore, eso **no demuestra que no exista** un circuito Hamiltoniano; por eso ejecutamos el algoritmo de búsqueda.


In [9]:
for numero_bloque in range(1, 6):
    imprimir_verificacion_hamiltoniana(numero_bloque)

resultados_hamiltonianos = analizar_circuitos_hamiltonianos()



VERIFICACIÓN HAMILTONIANA - BLOQUE 1

Farmacias del bloque:
  33 Calle Col Guillen: grado = 4
  SIMÁN PLAZA VICAY: grado = 3
  Simán Arenales: grado = 5
  Simán Calpules: grado = 4
  Simán Calpules 2: grado = 4
  Simán Ivihsa: grado = 5
  Simán Luisiana: grado = 3
  Simán Pradera: grado = 5
  Simán Santa Martha: grado = 3
  Simán Zip San Jose: grado = 2
  Town Center: grado = 3
  VILLA OLIMPICA: grado = 3

------------------------------------------------------------------------------------------
TEOREMA DE DIRAC
------------------------------------------------------------------------------------------
Numero de vertices: 12
Grado minimo:       2
n / 2:              6.0
Resultado: NO CUMPLE LA CONDICION DE DIRAC
Esto NO significa que no exista un circuito Hamiltoniano.

------------------------------------------------------------------------------------------
TEOREMA DE ORE
------------------------------------------------------------------------------------------
Resultado: NO CUMPLE L

## 10. Conclusión de la verificación Hamiltoniana

**Procedencia del algoritmo:** el problema de encontrar un circuito Hamiltoniano es NP completo  no existe un metodo general eficiente conocido por lo que se uso **backtracking** (busqueda exhaustiva con retroceso), el enfoque estandar para instancias pequeñas como estas (12 nodos por bloque). El algoritmo prueba vecinos no visitados en profundidad y retrocede cuando llega a un callejon sin salida, hasta encontrar un recorrido que visite las 12 sucursales una sola vez y regrese al inicio.

**Limitaciones:** el backtracking garantiza *encontrar un* circuito Hamiltoniano si existe, pero no necesariamente el de menor distancia (no es un algoritmo de optimizacion, solo de existencia/construccion). En el peor caso su complejidad es factorial, aunque aquí es tratable por el tamaño reducido de cada bloque (12 nodos). El circuito depende también de la sucursal usada como punto de partida y del orden en que se recorren los vecinos.


In [10]:
print(f"{'Bloque':<10}{'Dirac':<8}{'Ore':<8}{'Circuito valido':<18}{'Distancia prom.':<16}")
print("-"*60)

bloques_sin_dirac_ni_ore=0

for numero_bloque in range(1,6):
    bloque=f"Bloque {numero_bloque}"
    subgrafo=obtener_subgrafo_bloque(bloque)
    dirac=verificar_dirac(subgrafo)
    ore=verificar_ore(subgrafo)
    resultado=resultados_hamiltonianos[numero_bloque]

    if not dirac["cumple"] and not ore["cumple"]:
        bloques_sin_dirac_ni_ore+=1

    valido="Sí" if resultado["valido"] else "No"
    dist=formato_km(resultado["promedio"]) if resultado["valido"] else "N/D"

    print(f"{bloque:<10}{'Sí' if dirac['cumple'] else 'No':<8}{'Sí' if ore['cumple'] else 'No':<8}{valido:<18}{dist:<16}")

print()
if bloques_sin_dirac_ni_ore==5:
    print("Interpretación: NINGUN bloque cumple Dirac ni Ore, es decir, el teorema")
    print("no puede garantizar por si solo la existencia de un circuito Hamiltoniano")
    print("en ninguno de los 5 bloques. Sin embargo, el backtracking SI encontro un")
    print("circuito valido en los 5 casos. Esto confirma que Dirac y Ore son")
    print("condiciones SUFICIENTES pero no NECESARIAS: un grafo puede ser")
    print("Hamiltoniano sin cumplirlas, por eso fue indispensable programar la")
    print("busqueda por backtracking en lugar de basarse únicamente en los teoremas.")
else:
    print(f"Interpretación: {bloques_sin_dirac_ni_ore} de 5 bloques no cumplen ni Dirac ni Ore.")


Bloque    Dirac   Ore     Circuito valido   Distancia prom. 
------------------------------------------------------------
Bloque 1  No      No      Sí                23.61 km        
Bloque 2  No      No      Sí                19.27 km        
Bloque 3  No      No      Sí                22.25 km        
Bloque 4  No      No      Sí                30.18 km        
Bloque 5  No      No      Sí                12.90 km        

Interpretación: NINGUN bloque cumple Dirac ni Ore, es decir, el teorema
no puede garantizar por si solo la existencia de un circuito Hamiltoniano
en ninguno de los 5 bloques. Sin embargo, el backtracking SI encontro un
circuito valido en los 5 casos. Esto confirma que Dirac y Ore son
condiciones SUFICIENTES pero no NECESARIAS: un grafo puede ser
Hamiltoniano sin cumplirlas, por eso fue indispensable programar la
busqueda por backtracking en lugar de basarse únicamente en los teoremas.


## 10. Comparación de conectividad de los 3 candidatos

Función que recibe la letra de un candidato (`A`, `B` o `C`) y muestra su distancia hacia la sucursal de entrada de cada uno de los 5 bloques, ordenadas de la más cercana a la más lejana. Así se puede llamar la función cambiando el candidato y comparar el output directamente, sin necesidad de Dijkstra.


In [11]:
def comparar_candidato(letra):
    """Muestra, ordenadas de menor a mayor distancia, las distancias del
    candidato hacia la sucursal de entrada de cada uno de los 5 bloques."""
    letra=str(letra).upper()
    info=datos_candidato(letra)
    datos_bloques=Candidato_distancias[letra]

    #se ordenan los bloques del candidato de menor a mayor distancia promedio
    filas=sorted(datos_bloques.items(),key=lambda item:item[1]["promedio"])

    imprimir_linea()
    print(f"{info['nombre'].upper()} - {info['zona'].upper()}")
    print(f"Ubicación: {info['ubicacion']}")
    imprimir_linea()

    suma_total=0.0
    for bloque,datos in filas:
        print(f"\nBloque {bloque}: {datos['sucursal']}")
        print(f"   Ida:      {formato_km(datos['ida'])}")
        print(f"   Vuelta:   {formato_km(datos['vuelta'])}")
        print(f"   Promedio: {formato_km(datos['promedio'])}")
        suma_total+=datos["promedio"]

    print(f"\nBloque mas cercano al {info['nombre']}: {filas[0][1]['sucursal']} ({formato_km(filas[0][1]['promedio'])})")
    print(f"Bloque mas lejano al {info['nombre']}:  {filas[-1][1]['sucursal']} ({formato_km(filas[-1][1]['promedio'])})")
    print(f"Suma de distancias promedio a los 5 bloques: {formato_km(suma_total)}")

    return suma_total


## 11. Ejecutar la comparación y elegir el candidato con mejor conectividad

In [12]:
sumas={}

for letra in ["A","B","C"]:
    sumas[letra]=comparar_candidato(letra)
    print()

imprimir_linea()
print("COMPARACIÓN FINAL DE LOS 3 CANDIDATOS")
imprimir_linea()

for letra,suma in sorted(sumas.items(),key=lambda item:item[1]):
    print(f"Candidato {letra}: suma total de distancias promedio = {formato_km(suma)}")

mejor=min(sumas,key=sumas.get)
print(f"\nEl {nombre_candidato(mejor)} tiene la mejor conectividad general, "
      f"por tener la menor suma de distancias promedio hacia los 5 bloques.")


CANDIDATO A - ZONA NORTE
Ubicación: Sector La 105 Brigada

Bloque 5: Simán 3 Ave. - Centro SPS
   Ida:      2.20 km
   Vuelta:   2.40 km
   Promedio: 2.30 km

Bloque 3: Barandillas
   Ida:      2.20 km
   Vuelta:   3.00 km
   Promedio: 2.60 km

Bloque 2: Rancho El Coco
   Ida:      3.60 km
   Vuelta:   3.60 km
   Promedio: 3.60 km

Bloque 4: El Roble- Plaza Bogran
   Ida:      3.90 km
   Vuelta:   4.10 km
   Promedio: 4.00 km

Bloque 1: 33 Calle Col Guillen
   Ida:      8.80 km
   Vuelta:   9.00 km
   Promedio: 8.90 km

Bloque mas cercano al Candidato A: Simán 3 Ave. - Centro SPS (2.30 km)
Bloque mas lejano al Candidato A:  33 Calle Col Guillen (8.90 km)
Suma de distancias promedio a los 5 bloques: 21.40 km

CANDIDATO B - ZONA SUR
Ubicación: Zona industrial, Sector el Polvorín

Bloque 1: 33 Calle Col Guillen
   Ida:      1.20 km
   Vuelta:   0.90 km
   Promedio: 1.05 km

Bloque 5: Simán 3 Ave. - Centro SPS
   Ida:      6.40 km
   Vuelta:   5.90 km
   Promedio: 6.15 km

Bloque 3: Barand